# AP01 – Ergänzungen

In [1]:
# Libraries
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# Current working directory
print('Arbeitsverzeichnis:', Path.cwd())

#NICHTS ANPASSEN: Imports und Path.cwd()
#ERKENNTNIS: Relative Dateipfade beziehen sich auf das Arbeitsverzeichnis des Notebooks

Arbeitsverzeichnis: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01/AP01 Mock Exam


In [2]:
user = 'pgadmin'                                                                #PostgreSQL-Benutzer festlegen
password = 'geheim'                                                             #Passwort des Benutzers festlegen
host = 'localhost'                                                              #Adresse des PostgreSQL-Servers festlegen
port = '5432'                                                                   #Port des PostgreSQL-Servers festlegen
database = 'postgres'                                                           #Gewünschte Datenbank festlegen

db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'  #Verbindungsadresse aus den Zugangsdaten zusammensetzen

engine = create_engine(db_connection_url)                                       #Engine für den Datenbankzugriff erstellen

df_spickap01 = pd.read_sql_query("""
    SELECT current_database() AS database_name,                                 --Namen der verbundenen Datenbank anzeigen
           (SELECT COUNT(*) FROM bicycles) AS bicycles_count,                   --Zeilen der Fahrradtabelle zählen
           (SELECT COUNT(*) FROM sales) AS sales_count,                         --Zeilen der Verkaufstabelle zählen
           (SELECT COUNT(*) FROM servicerecords) AS services_count;             --Zeilen der Servicetabelle zählen
""", con=engine)                                                                #SQL ausführen und Prüfergebnis als DataFrame speichern

engine.dispose()                                                                #Vom Verbindungspool gehaltene Verbindungen freigeben
display(df_spickap01)                                                           #Datenbankname und Tabellenanzahlen anzeigen

#ANPASSEN Zugangsdaten nur falls die Prüfungsumgebung andere Angaben vorgibt
#ANPASSEN Tabellennamen in den drei Unterabfragen gemäss Prüfungsdatensatz

,database_name,bicycles_count,sales_count,services_count
0,postgres,2000,684,7500


## Rules

## Ergänzungen – Regeln und Merksätze

### Berechnungen, Datentypen und Texte
- CAST und `::` wandeln Werte um, im SELECT bleiben die gespeicherten Daten unverändert
- `ROUND(..., 2)` benötigt NUMERIC, bspw. `ROUND(CAST(price AS NUMERIC), 2)`
- Ganzzahldivision schneidet Nachkommastellen ab, `5 / 2` ergibt 2 und `5 / 2.0` ergibt 2.5
- `COALESCE(spalte, ersatzwert)` ersetzt NULL, aber keinen leeren Text
- CASE verwendet den ersten passenden WHEN-Fall, ohne Treffer und ohne ELSE entsteht NULL
- `||` verbindet Texte, bei einem NULL-Teil wird das Ergebnis NULL
- SUBSTRING zählt Positionen ab 1

### Unterabfragen
- Eine Unterabfrage nach `=`, `>` oder `<` muss genau eine Spalte und höchstens eine Zeile liefern
- IN prüft auf passende Werte, EXISTS nur auf mindestens eine passende Zeile, beide vervielfachen die äussere Zeile nicht
- Bei NOT IN können NULL-Werte der Unterabfrage Treffer verhindern, NULL ausschliessen oder NOT EXISTS verwenden
- `> ANY` bedeutet grösser als mindestens ein Vergleichswert, `> ALL` grösser als jeden Vergleichswert
- Bei leerer Unterabfrage ist ANY falsch und ALL wahr

### Fensterfunktionen und UNION
- `OVER ()` berechnet über alle betrachteten Zeilen, `PARTITION BY` getrennt nach Gruppen, Einzelzeilen bleiben erhalten
- ROW_NUMBER vergibt eindeutige Nummern, RANK vergibt bei Gleichständen denselben Rang mit anschliessenden Lücken
- ORDER BY innerhalb von OVER steuert die Fensterfunktion, erst das äussere ORDER BY garantiert die Ausgabereihenfolge
- UNION entfernt doppelte vollständige Ergebniszeilen, UNION ALL behält sie
- UNION-Abfragen benötigen gleich viele Spalten mit kompatiblen Datentypen in derselben Reihenfolge

### Regex
- `~` berücksichtigt Gross- und Kleinschreibung, `~*` ignoriert sie, `!~` und `!~*` prüfen auf keinen Treffer
- Ohne Begrenzung genügt ein Teiltreffer, `^...$` prüft den gesamten Text
- `.` steht für ein beliebiges Zeichen, `\.` oder `[.]` für einen tatsächlichen Punkt
- `[0-9]` findet eine Ziffer, `[^0-9]` ein Zeichen das keine Ziffer ist
- `*` bedeutet nullmal oder öfter, `+` mindestens einmal, `?` nullmal oder einmal
- `{n}` genau n-mal, `{n,m}` n- bis m-mal, `{n,}` mindestens n-mal
- Wiederholungen gelten für das unmittelbar vorherige Element, Klammern gruppieren und `|` trennt Alternativen
- Regex im SELECT liefert TRUE, FALSE oder NULL, in WHERE bleiben nur TRUE-Zeilen
- Für Regex-Backslashes im Python-Code `r"""..."""` verwenden

### Daten und Tabellen verändern
- INSERT fügt Zeilen ein, Werte müssen zur Spaltenreihenfolge und zu den Datentypen passen
- UPDATE und DELETE ohne WHERE betreffen alle Zeilen, die Auswahl vorher mit SELECT prüfen
- Wiederholte prozentuale UPDATEs verändern bereits angepasste Werte erneut
- Bei UPDATE FROM sollte pro Zielzeile höchstens eine Quellzeile passen
- Änderungen über SQLAlchemy bspw. mit `with engine.begin() as connection` ausführen, bei Erfolg wird automatisch gespeichert und bei Fehler zurückgerollt
- Temporäre Tabellen gelten nur für die aktuelle Datenbanksitzung, zusammengehörende Schritte über dieselbe Verbindung ausführen
- ALTER ändert die Tabellenstruktur, DELETE entfernt Zeilen und DROP TABLE entfernt Daten inkl. Tabellenstruktur

## Queries

### EXTRACT — Retrieve each bicycle sale with its bicycle ID, make, model, sale date and sale year. Display the year as an integer
- `INNER JOIN` verbindet Fahrräder und Verkäufe über `bicycleid`, `CAST(EXTRACT(YEAR FROM s.saledate) AS INTEGER)` gibt das Verkaufsjahr als Ganzzahl aus
- Statt `YEAR` sind bspw. `MONTH` oder `DAY` möglich, das ursprüngliche Verkaufsdatum bleibt unverändert
- Format: `SELECT Tabelle1.*, Tabelle2.* FROM Tabelle1 INNER JOIN Tabelle2 ON Tabelle1.GemeinsameSpalte = Tabelle2.GemeinsameSpalte;`

In [3]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT b.bicycleid, b.make, b.model, s.saledate,                            --Fahrrad und Verkaufsdatum anzeigen
           CAST(EXTRACT(YEAR FROM s.saledate) AS INTEGER) AS sale_year          --Jahr aus dem Datum lesen und als Ganzzahl ausgeben
    FROM bicycles AS b                                                          --Fahrradtabelle verwenden
    INNER JOIN sales AS s ON b.bicycleid = s.bicycleid                          --Verkaufsdatum über die Fahrrad-ID ergänzen
    ORDER BY s.saledate, b.bicycleid;                                           --Chronologisch und bei gleichem Datum nach Fahrrad-ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,saledate,sale_year
0,C9PGTJQX18,Cannondale,SuperSix,2019-08-29,2019
1,C8QUWFU87V,Santa Cruz,Bronson,2019-09-06,2019
2,CJSMB9CVGE,Bianchi,Infinito,2019-09-13,2019
3,CP64NZ2VJ0,Giant,Talon,2019-09-14,2019
4,C2WFIK6WEP,Merida,Scultura,2019-09-15,2019
...,...,...,...,...,...
679,C6P05HHSPM,Giant,TCR,2024-08-11,2024
680,CMMCJADVB1,Cannondale,Trail,2024-08-14,2024
681,CSX9QRNFZE,Scott,Aspect,2024-08-14,2024
682,CWE0XO1VM0,Giant,TCR,2024-08-16,2024


### EXTRACT + GROUP BY — Determine the average cost per year for services described as "Gear shifting adjustment". Display year and average_service_costs, rounded to two decimal places and sorted by average_service_costs in descending order
- `WHERE` filtert auf den gewünschten Service, `GROUP BY CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER)` bildet eine Gruppe pro Jahr
- `AVG(cost)` berechnet den Durchschnitt je Jahr, `ROUND(CAST(... AS NUMERIC), 2)` rundet die Ausgabe und `ORDER BY average_service_costs DESC` zeigt den höchsten Durchschnitt zuerst

In [4]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER) AS year,             --Jahr des Services auslesen und als Ganzzahl ausgeben
           ROUND(CAST(AVG(cost) AS NUMERIC), 2) AS average_service_costs        --Durchschnittliche Kosten pro Jahr auf zwei Stellen runden
    FROM servicerecords                                                         --Serviceeinträge des Bike Shops verwenden
    WHERE servicedescription = 'Gear shifting adjustment'                       --Nur diese Servicebeschreibung berücksichtigen
    GROUP BY CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER)                    --Alle passenden Services desselben Jahres zusammenfassen
    ORDER BY average_service_costs DESC;                                        --Höchsten durchschnittlichen Servicepreis zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,year,average_service_costs
0,2019,25.83
1,2021,25.71
2,2020,25.40
3,2022,25.30
4,2023,25.07
5,2024,24.59


### CAST — Retrieve each bicycle ID and its year, both as the original number and as text. Name the additional column year_text
- `CAST(year AS TEXT) AS year_text` gibt das Baujahr zusätzlich als Text aus, die ursprüngliche Spalte bleibt unverändert
- Format: `CAST(spalte AS datentyp)` wandelt einen Wert in den gewünschten Datentyp um, sofern die Umwandlung möglich ist

In [5]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, year,                                                     --ID und ursprüngliches Baujahr auswählen
           CAST(year AS TEXT) AS year_text                                      --Baujahr nur für die Ausgabe in Text umwandeln
    FROM bicycles;                                                              --Der gespeicherte Datentyp wird dadurch nicht verändert
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,year,year_text
0,C7ZGWFLBUV,2019,2019
1,C875HZ1AG6,2021,2021
2,CDONEWSWFP,2021,2021
3,C2UQZP803C,2023,2023
4,C2WFIK6WEP,2020,2020
...,...,...,...
1995,CLH919EIKV,2020,2020
1996,CP3F48BE9Y,2022,2022
1997,CHBXQ3SSNU,2019,2019
1998,CP5CTME2ZD,2021,2021


### :: — Retrieve each bicycle ID and its year as text. Use PostgreSQL shorthand casting and name the result year_text
- `year::TEXT AS year_text` gibt das Baujahr als Text aus, gleichbedeutend mit `CAST(year AS TEXT)`
- Format: `spalte::datentyp`, der gespeicherte Spaltentyp bleibt unverändert

In [6]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                                                           --Fahrrad-ID auswählen
           year::TEXT AS year_text                                              --PostgreSQL-Kurzform für CAST(year AS TEXT)
    FROM bicycles;                                                              --Nur die Abfrageausgabe umwandeln, nicht die Originalspalte
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,year_text
0,C7ZGWFLBUV,2019
1,C875HZ1AG6,2021
2,CDONEWSWFP,2021
3,C2UQZP803C,2023
4,C2WFIK6WEP,2020
...,...,...
1995,CLH919EIKV,2020
1996,CP3F48BE9Y,2022
1997,CHBXQ3SSNU,2019
1998,CP5CTME2ZD,2021


### ROUND + calculation — Retrieve each bicycle ID, make, model and price, and calculate a price reduced by 10%. Round discounted_price to two decimal places
- `price * 0.90` berechnet den Preis nach 10% Rabatt, der gespeicherte Originalpreis bleibt unverändert
- `ROUND(CAST(price * 0.90 AS NUMERIC), 2) AS discounted_price` rundet den neuen Preis auf zwei Dezimalstellen und benennt die Ergebnisspalte

In [7]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price,                                       --Originaldaten und ursprünglichen Preis anzeigen
           ROUND(CAST(price * 0.90 AS NUMERIC), 2) AS discounted_price          --Zehn Prozent Rabatt berechnen und auf zwei Stellen runden
    FROM bicycles;                                                              --Berechnung nur ausgeben, gespeicherte Preise bleiben unverändert
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price,discounted_price
0,C7ZGWFLBUV,Trek,Domane,5066.94,4560.25
1,C875HZ1AG6,Merida,Reacto,5529.38,4976.44
2,CDONEWSWFP,Merida,Big Nine,1518.33,1366.50
3,C2UQZP803C,Trek,Marlin,5038.99,4535.09
4,C2WFIK6WEP,Merida,Scultura,1548.19,1393.37
...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2376.02,2138.42
1996,CP3F48BE9Y,Merida,Scultura,2833.36,2550.02
1997,CHBXQ3SSNU,Merida,Scultura,1629.91,1466.92
1998,CP5CTME2ZD,Trek,Marlin,6176.55,5558.90


### COALESCE — Retrieve each bicycle ID, make, model and status. Display "Unknown" when the status is missing and name that output column status_display
- `COALESCE(status, 'Unknown') AS status_display` zeigt bei NULL den Ersatztext Unknown, vorhandene Werte bleiben erhalten
- Format: `COALESCE(spalte, ersatzwert)` liefert den ersten nicht-NULL-Wert, leerer Text `''` wird nicht ersetzt und die Originaldaten bleiben unverändert

In [8]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model,                                              --Fahrraddaten auswählen
           COALESCE(status, 'Unknown') AS status_display                        --Ersten nicht-NULL-Wert ausgeben, leeren Text nicht ersetzen
    FROM bicycles;                                                              --Nur die Ausgabe anpassen, Originaldaten bleiben unverändert
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,status_display
0,C7ZGWFLBUV,Trek,Domane,In Maintenance
1,C875HZ1AG6,Merida,Reacto,Sold
2,CDONEWSWFP,Merida,Big Nine,In Maintenance
3,C2UQZP803C,Trek,Marlin,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,Sold
...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,Available
1996,CP3F48BE9Y,Merida,Scultura,Available
1997,CHBXQ3SSNU,Merida,Scultura,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,In Maintenance


### CASE — Retrieve each bicycle ID and price, and classify the price as "Unknown" if missing, "Budget" below 2000, "Mid-range" from 2000 to 4000 inclusive, or "Premium" above 4000. Name the new column price_category
- `CASE` prüft die Bedingungen der Reihe nach, die erste erfüllte Bedingung bestimmt die Kategorie. NULL zuerst prüfen, danach die Preisgrenzen
- Format: `CASE WHEN bedingung THEN ergebnis ELSE sonstiges END AS price_category`, ohne passenden Fall und ohne ELSE entsteht NULL

In [9]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, price,                                                    --Fahrrad-ID und ursprünglichen Preis anzeigen
           CASE                                                                 --Fallunterscheidung beginnen; erster passender WHEN-Zweig entscheidet
               WHEN price IS NULL THEN 'Unknown'                                --Fehlenden Preis ausdrücklich behandeln
               WHEN price < 2000 THEN 'Budget'                                  --Preise unter 2000 zuordnen
               WHEN price <= 4000 THEN 'Mid-range'                              --Nach dem vorigen Zweig bleiben hier Preise von 2000 bis 4000
               ELSE 'Premium'                                                   --Übrige Preise über 4000 zuordnen
           END AS price_category                                                --Fallunterscheidung beenden und Ergebnisspalte benennen
    FROM bicycles;                                                              --Keine Preiswerte in der Datenbank ändern
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,price,price_category
0,C7ZGWFLBUV,5066.94,Premium
1,C875HZ1AG6,5529.38,Premium
2,CDONEWSWFP,1518.33,Budget
3,C2UQZP803C,5038.99,Premium
4,C2WFIK6WEP,1548.19,Budget
...,...,...,...
1995,CLH919EIKV,2376.02,Mid-range
1996,CP3F48BE9Y,2833.36,Mid-range
1997,CHBXQ3SSNU,1629.91,Budget
1998,CP5CTME2ZD,6176.55,Premium


### || — Retrieve each bicycle ID and combine make and model into one space-separated column named bicycle_name
- `make || ' ' || model AS bicycle_name` verbindet Marke und Modell mit einem Leerzeichen, die Originalspalten bleiben unverändert
- Format: `spalte1 || 'trennzeichen' || spalte2`, ist einer der Werte NULL, wird das gesamte Ergebnis NULL

In [10]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid,                                                           --Fahrrad-ID auswählen
           make || ' ' || model AS bicycle_name                                 --Marke, Leerzeichen und Modell zu einem Text verbinden
    FROM bicycles;                                                              --Bei NULL in einem Bestandteil ist die gesamte Verkettung NULL
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,bicycle_name
0,C7ZGWFLBUV,Trek Domane
1,C875HZ1AG6,Merida Reacto
2,CDONEWSWFP,Merida Big Nine
3,C2UQZP803C,Trek Marlin
4,C2WFIK6WEP,Merida Scultura
...,...,...
1995,CLH919EIKV,Trek Emonda
1996,CP3F48BE9Y,Merida Scultura
1997,CHBXQ3SSNU,Merida Scultura
1998,CP5CTME2ZD,Trek Marlin


### UPPER — Retrieve each bicycle ID, its original make and the make in uppercase. Name the additional column make_upper
- `UPPER(make) AS make_upper` gibt die Marke in Grossbuchstaben aus, die Originalspalte bleibt unverändert
- Format: `UPPER(spalte)`, bspw. Bianchi → BIANCHI

In [11]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make,                                                     --Fahrrad-ID und Originalmarke auswählen
           UPPER(make) AS make_upper                                            --Marke in Grossbuchstaben ausgeben
    FROM bicycles;                                                              --Originalmarke bleibt gespeichert wie zuvor
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,make_upper
0,C7ZGWFLBUV,Trek,TREK
1,C875HZ1AG6,Merida,MERIDA
2,CDONEWSWFP,Merida,MERIDA
3,C2UQZP803C,Trek,TREK
4,C2WFIK6WEP,Merida,MERIDA
...,...,...,...
1995,CLH919EIKV,Trek,TREK
1996,CP3F48BE9Y,Merida,MERIDA
1997,CHBXQ3SSNU,Merida,MERIDA
1998,CP5CTME2ZD,Trek,TREK


### LOWER — Retrieve each bicycle ID, its original make and the make in lowercase. Name the additional column make_lower
- `LOWER(make) AS make_lower` gibt die Marke in Kleinbuchstaben aus, die Originalspalte bleibt unverändert
- Format: `LOWER(spalte)`, bspw. Bianchi → bianchi

In [12]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make,                                                     --Fahrrad-ID und Originalmarke auswählen
           LOWER(make) AS make_lower                                            --Marke in Kleinbuchstaben ausgeben
    FROM bicycles;                                                              --Gross-/Kleinschreibung nur in der Ausgabe ändern
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,make_lower
0,C7ZGWFLBUV,Trek,trek
1,C875HZ1AG6,Merida,merida
2,CDONEWSWFP,Merida,merida
3,C2UQZP803C,Trek,trek
4,C2WFIK6WEP,Merida,merida
...,...,...,...
1995,CLH919EIKV,Trek,trek
1996,CP3F48BE9Y,Merida,merida
1997,CHBXQ3SSNU,Merida,merida
1998,CP5CTME2ZD,Trek,trek


### LENGTH — Retrieve each bicycle ID, model and the number of characters in the model name. Name the additional column model_length and sort by it in descending order
- `LENGTH(model) AS model_length` zählt die Zeichen inkl. Leerzeichen, bei NULL entsteht NULL
- Format: `LENGTH(spalte)`, `ORDER BY model_length DESC NULLS LAST` zeigt die längsten Modellnamen zuerst und fehlende Werte zuletzt

In [13]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, model,                                                    --Fahrrad-ID und Modell auswählen
           LENGTH(model) AS model_length                                        --Zeichen inklusive Leerzeichen zählen; NULL bleibt NULL
    FROM bicycles                                                               --Fahrradtabelle verwenden
    ORDER BY model_length DESC NULLS LAST, bicycleid;                           --Längste Modellnamen zuerst; bei Gleichstand nach ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,model,model_length
0,C15T1IUY8R,Rockhopper,10
1,C19I6367PV,Rockhopper,10
2,C1AP0CNR2L,Rockhopper,10
3,C2HX6YP3UN,Rockhopper,10
4,C34OXLI9KY,Rockhopper,10
...,...,...,...
1995,CXSPK20YP0,R5,2
1996,CXW9GFQT40,R5,2
1997,CZENR3UUJU,R5,2
1998,CZMECIKHMX,R5,2


### REPLACE — Retrieve each bicycle ID and status, and show a version in which "In Maintenance" is replaced by "Maintenance". Name the additional column status_display
- `REPLACE(status, 'In Maintenance', 'Maintenance') AS status_display` ersetzt den Text nur in der Ausgabe, die Originaldaten bleiben unverändert
- Format: `REPLACE(spalte, 'suchtext', 'ersatztext')`, ersetzt alle Vorkommen und berücksichtigt Gross- und Kleinschreibung

In [14]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, status,                                                   --Fahrrad-ID und Originalstatus anzeigen
           REPLACE(status, 'In Maintenance', 'Maintenance') AS status_display   --Exakten Text ersetzen; Gross-/Kleinschreibung beachten
    FROM bicycles;                                                              --Nur Ausgabetext ändern, nicht den gespeicherten Status
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,status,status_display
0,C7ZGWFLBUV,In Maintenance,Maintenance
1,C875HZ1AG6,Sold,Sold
2,CDONEWSWFP,In Maintenance,Maintenance
3,C2UQZP803C,In Maintenance,Maintenance
4,C2WFIK6WEP,Sold,Sold
...,...,...,...
1995,CLH919EIKV,Available,Available
1996,CP3F48BE9Y,Available,Available
1997,CHBXQ3SSNU,In Maintenance,Maintenance
1998,CP5CTME2ZD,In Maintenance,Maintenance


### SUBSTRING — Retrieve each bicycle ID, model and the first three characters of its model name. Name the additional column model_prefix
- `SUBSTRING(model FROM 1 FOR 3) AS model_prefix` gibt die ersten drei Zeichen des Modellnamens aus, die erste Position ist 1
- Format: `SUBSTRING(spalte FROM startposition FOR anzahl_zeichen)`, die Originaldaten bleiben unverändert

In [15]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, model,                                                    --Fahrrad-ID und Originalmodell auswählen
           SUBSTRING(model FROM 1 FOR 3) AS model_prefix                        --Ab Position 1 drei Zeichen entnehmen; SQL beginnt bei 1
    FROM bicycles;                                                              --Kurze Modellnamen liefern entsprechend weniger Zeichen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,model,model_prefix
0,C7ZGWFLBUV,Domane,Dom
1,C875HZ1AG6,Reacto,Rea
2,CDONEWSWFP,Big Nine,Big
3,C2UQZP803C,Marlin,Mar
4,C2WFIK6WEP,Scultura,Scu
...,...,...,...
1995,CLH919EIKV,Emonda,Emo
1996,CP3F48BE9Y,Scultura,Scu
1997,CHBXQ3SSNU,Scultura,Scu
1998,CP5CTME2ZD,Marlin,Mar


### AND + OR + parentheses — Retrieve bicycle ID, make, model and price for bicycles made by Bianchi or Trek, but only if their price is greater than 3000
- `(make = 'Bianchi' OR make = 'Trek') AND price > 3000` verlangt für beide Marken einen Preis über 3000
- `AND` bindet stärker als `OR`, ohne Klammern würden alle Bianchi-Fahrräder unabhängig vom Preis ausgewählt

In [16]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Gewünschte Fahrradspalten auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE (make = 'Bianchi' OR make = 'Trek')                                   --Klammern fassen die beiden Markenalternativen zusammen
      AND price > 3000;                                                         --Preisbedingung gilt für beide Marken, AND bindet sonst stärker als OR
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C2UQZP803C,Trek,Marlin,5038.99
2,C2ZEK0YPD9,Bianchi,Oltre,7382.81
3,CE5XORIKH7,Trek,Emonda,7600.96
4,CB46L6E7V3,Trek,Emonda,4396.75
...,...,...,...,...
267,CB4PK8PLSN,Bianchi,Oltre,7765.04
268,C1V12YL0CG,Trek,Domane,7497.23
269,CFFTG1MNW0,Bianchi,Aria,5939.92
270,C6DHR3STN7,Trek,Marlin,6624.90


### Subquery: overall average — Retrieve bicycle ID, make, model and price for bicycles priced above the overall average bicycle price. Sort by price in descending order
- Die Unterabfrage `(SELECT AVG(price) FROM bicycles)` berechnet den Gesamtdurchschnitt, `WHERE price > (...)` wählt nur Fahrräder darüber aus
- Format: `WHERE spalte > (SELECT AVG(spalte) FROM tabelle)`, `ORDER BY price DESC` zeigt die teuersten Fahrräder zuerst

In [17]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrraddaten der äusseren Abfrage auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE price > (SELECT AVG(price) FROM bicycles)                             --Innere Abfrage liefert einen Vergleichswert: den Gesamtdurchschnitt
    ORDER BY price DESC, bicycleid;                                             --Höchste Preise zuerst, bei gleichem Preis nach ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C187CGZK39,Trek,Marlin,7995.90
1,CENB0TLQKE,Bianchi,Oltre,7989.74
2,CJSLUUOM7Z,Santa Cruz,Hightower,7984.31
3,CLQCGBXCAG,Merida,Scultura,7972.88
4,CTVKH7S3PZ,Trek,Marlin,7972.58
...,...,...,...,...
984,CKA8RS2OQG,Trek,Emonda,4400.88
985,C70A8JPBP6,Scott,Aspect,4399.97
986,CB46L6E7V3,Trek,Emonda,4396.75
987,CM6ANT1O7T,Cervelo,Soloist,4392.99


### IN with subquery — Retrieve bicycle ID, make, model and price for bicycles that appear in at least one sales record
- `WHERE bicycleid IN (SELECT bicycleid FROM sales)` wählt Fahrräder aus, deren ID in mindestens einem Verkauf vorkommt
- Format: `WHERE spalte IN (SELECT vergleichsspalte FROM tabelle)`, mehrere passende Verkäufe vervielfachen die Fahrradzeile nicht

In [18]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrraddaten auswählen
    FROM bicycles                                                               --Jedes Fahrrad der äusseren Tabelle betrachten
    WHERE bicycleid IN (                                                        --ID muss in der Ergebnismenge der Unterabfrage enthalten sein
        SELECT bicycleid                                                        --Unterabfrage liefert genau eine Spalte
        FROM sales                                                              --Fahrrad-IDs aus den Verkaufsdatensätzen verwenden
    );                                                                          --Mehrere Verkäufe vervielfachen die Zeile der äusseren Tabelle nicht
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C875HZ1AG6,Merida,Reacto,5529.38
1,C2WFIK6WEP,Merida,Scultura,1548.19
2,CGPCXD8911,Scott,Aspect,1715.48
3,C3U71M5BP5,Specialized,Rockhopper,7904.79
4,CLD9BSTE3H,Specialized,Rockhopper,5712.21
...,...,...,...,...
679,CALG7KLJHE,Trek,Domane,6629.37
680,C78URTU7TV,Specialized,Rockhopper,7375.79
681,C1V12YL0CG,Trek,Domane,7497.23
682,C6OURRJ88X,Specialized,Rockhopper,4208.90


### NOT IN with subquery — Retrieve bicycle ID, make, model and price for bicycles that do not appear in any sales record. Exclude missing IDs from the subquery
- `NOT IN` wählt Fahrräder aus, deren ID in keinem Verkauf vorkommt, `WHERE bicycleid IS NOT NULL` entfernt fehlende IDs aus der Unterabfrage
- Format: `WHERE spalte NOT IN (SELECT vergleichsspalte FROM tabelle WHERE vergleichsspalte IS NOT NULL)`, NULL in der Unterabfrage kann sonst sämtliche Treffer verhindern

In [19]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrraddaten auswählen
    FROM bicycles                                                               --Jedes Fahrrad der äusseren Tabelle betrachten
    WHERE bicycleid NOT IN (                                                    --Nur IDs behalten, die in der Unterabfrage fehlen
        SELECT bicycleid                                                        --Verkaufte Fahrrad-IDs auswählen
        FROM sales                                                              --Verkaufstabelle verwenden
        WHERE bicycleid IS NOT NULL                                             --NULL ausschliessen, ein NULL in NOT IN kann sonst Treffer verhindern
    );                                                                          --Unterabfrage liefert ausschliesslich vorhandene IDs
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,CDONEWSWFP,Merida,Big Nine,1518.33
2,C2UQZP803C,Trek,Marlin,5038.99
3,C4RQCO25I9,Cube,Reaction,3339.27
4,C9LPTOS9KE,Cannondale,Synapse,5399.61
...,...,...,...,...
1311,CLH919EIKV,Trek,Emonda,2376.02
1312,CP3F48BE9Y,Merida,Scultura,2833.36
1313,CHBXQ3SSNU,Merida,Scultura,1629.91
1314,CP5CTME2ZD,Trek,Marlin,6176.55


### EXISTS — Retrieve bicycle ID, make, model and price for bicycles with at least one sales record. Use EXISTS
- `EXISTS` prüft für jedes Fahrrad, ob mind. ein Verkauf mit derselben Fahrrad-ID existiert, mehrere Verkäufe vervielfachen die Fahrradzeile nicht
- Format: `WHERE EXISTS (SELECT 1 FROM sales AS s WHERE s.bicycleid = b.bicycleid)`, `b` bezeichnet die äussere Fahrradtabelle und `SELECT 1` zählt nichts

In [20]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT b.bicycleid, b.make, b.model, b.price                                --Fahrraddaten der äusseren Tabelle auswählen
    FROM bicycles AS b                                                          --b ist der Kurzname der äusseren Fahrradtabelle
    WHERE EXISTS (                                                              --Prüfen, ob mind. ein passender Datensatz existiert
        SELECT 1                                                                --1 ist nur ein Platzhalter; EXISTS zählt keine Zeilen
        FROM sales AS s                                                         --Passenden Verkaufsdatensatz suchen
        WHERE s.bicycleid = b.bicycleid                                         --Unterabfrage mit dem aktuell betrachteten Fahrrad verknüpfen
    );                                                                          --Auch bei mehreren Verkäufen jedes Fahrrad nur einmal auswählen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C875HZ1AG6,Merida,Reacto,5529.38
1,C2WFIK6WEP,Merida,Scultura,1548.19
2,CGPCXD8911,Scott,Aspect,1715.48
3,C3U71M5BP5,Specialized,Rockhopper,7904.79
4,CLD9BSTE3H,Specialized,Rockhopper,5712.21
...,...,...,...,...
679,CALG7KLJHE,Trek,Domane,6629.37
680,C78URTU7TV,Specialized,Rockhopper,7375.79
681,C1V12YL0CG,Trek,Domane,7497.23
682,C6OURRJ88X,Specialized,Rockhopper,4208.90


### NOT EXISTS — Retrieve bicycle ID, make, model and price for bicycles without any sales record. Use NOT EXISTS
- `NOT EXISTS` wählt Fahrräder aus, für die kein Verkauf mit derselben Fahrrad-ID existiert
- Format: `WHERE NOT EXISTS (SELECT 1 FROM sales AS s WHERE s.bicycleid = b.bicycleid)`, `b` bezeichnet die äussere Fahrradtabelle und NULL-Werte in `sales.bicycleid` blockieren die Auswahl nicht

In [21]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT b.bicycleid, b.make, b.model, b.price                                --Fahrraddaten auswählen
    FROM bicycles AS b                                                          --Jedes Fahrrad der äusseren Tabelle betrachten
    WHERE NOT EXISTS (                                                          --Nur Fahrräder ohne passenden Verkaufsdatensatz behalten
        SELECT 1                                                                --Entscheidend ist die Existenz eines Treffers, nicht der Wert 1
        FROM sales AS s                                                         --Verkaufstabelle durchsuchen
        WHERE s.bicycleid = b.bicycleid                                         --Nach Verkäufen des aktuellen Fahrrads suchen
    );                                                                          --NULL-Werte in anderen Verkaufszeilen verhindern diese Prüfung nicht
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,CDONEWSWFP,Merida,Big Nine,1518.33
2,C2UQZP803C,Trek,Marlin,5038.99
3,C4RQCO25I9,Cube,Reaction,3339.27
4,C9LPTOS9KE,Cannondale,Synapse,5399.61
...,...,...,...,...
1311,CLH919EIKV,Trek,Emonda,2376.02
1312,CP3F48BE9Y,Merida,Scultura,2833.36
1313,CHBXQ3SSNU,Merida,Scultura,1629.91
1314,CP5CTME2ZD,Trek,Marlin,6176.55


### ANY — Retrieve bicycle ID, make, model and price for bicycles that cost more than at least one Bianchi bicycle with a known price
- `> ANY` verlangt einen Preis über mind. einem bekannten Bianchi-Preis, bei Vergleichspreisen von 2000 und 4000 genügt ein Preis über 2000
- Format: `WHERE price > ANY (SELECT price FROM bicycles WHERE make = 'Bianchi' AND price IS NOT NULL)`, bei leerer Unterabfrage wird kein Fahrrad ausgewählt

In [22]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrraddaten auswählen
    FROM bicycles                                                               --Alle Marken der äusseren Tabelle berücksichtigen
    WHERE price > ANY (                                                         --Mind. einer der Vergleiche muss zutreffen
        SELECT price                                                            --Vergleichspreise auswählen
        FROM bicycles                                                           --Bianchi-Preise aus derselben Tabelle lesen
        WHERE make = 'Bianchi' AND price IS NOT NULL                            --Nur bekannte Bianchi-Preise als Vergleich verwenden
    );                                                                          --Bei leerer Unterabfrage ist > ANY FALSE
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1959,CLH919EIKV,Trek,Emonda,2376.02
1960,CP3F48BE9Y,Merida,Scultura,2833.36
1961,CHBXQ3SSNU,Merida,Scultura,1629.91
1962,CP5CTME2ZD,Trek,Marlin,6176.55


### ALL — Retrieve bicycle ID, make, model and price for bicycles that cost more than every Bianchi bicycle with a known price
- `> ALL` verlangt einen Preis über jedem bekannten Bianchi-Preis, bei Vergleichspreisen von 2000 und 4000 muss der Preis über 4000 liegen
- Format: `WHERE price > ALL (SELECT price FROM bicycles WHERE make = 'Bianchi' AND price IS NOT NULL)`, bei leerer Unterabfrage ist die Bedingung für alle Fahrräder erfüllt

In [23]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrraddaten auswählen
    FROM bicycles                                                               --Alle Marken der äusseren Tabelle berücksichtigen
    WHERE price > ALL (                                                         --Jeder Vergleich mit einem zurückgegebenen Preis muss zutreffen
        SELECT price                                                            --Vergleichspreise auswählen
        FROM bicycles                                                           --Bianchi-Preise aus derselben Tabelle lesen
        WHERE make = 'Bianchi' AND price IS NOT NULL                            --NULL aus der Vergleichsmenge ausschliessen
    );                                                                          --Bei leerer Unterabfrage ist > ALL TRUE; hier sind Bianchi-Preise vorhanden
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C187CGZK39,Trek,Marlin,7995.9


### AVG OVER — Retrieve every bicycle ID, make, model and price, and show the overall average price in an additional column named overall_average_price. Keep every bicycle row
- `AVG(price) OVER ()` ergänzt den Gesamtdurchschnitt in jeder Zeile, alle Fahrräder bleiben erhalten und NULL-Preise werden bei der Berechnung ignoriert
- Format: `AVG(spalte) OVER () AS alias`, anders als bei einer normalen Aggregation werden die Einzelzeilen nicht zusammengefasst

In [24]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price,                                          --Einzelne Fahrradzeilen beibehalten
           ROUND(CAST(AVG(price) OVER () AS NUMERIC), 2) AS overall_average_price  --Gesamtdurchschnitt als zusätzliche Spalte in jeder Zeile anzeigen
    FROM bicycles;                                                                 --OVER () fasst die Zeilen im Gegensatz zu GROUP BY nicht zusammen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price,overall_average_price
0,C7ZGWFLBUV,Trek,Domane,5066.94,4382.23
1,C875HZ1AG6,Merida,Reacto,5529.38,4382.23
2,CDONEWSWFP,Merida,Big Nine,1518.33,4382.23
3,C2UQZP803C,Trek,Marlin,5038.99,4382.23
4,C2WFIK6WEP,Merida,Scultura,1548.19,4382.23
...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2376.02,4382.23
1996,CP3F48BE9Y,Merida,Scultura,2833.36,4382.23
1997,CHBXQ3SSNU,Merida,Scultura,1629.91,4382.23
1998,CP5CTME2ZD,Trek,Marlin,6176.55,4382.23


### PARTITION BY — Retrieve every bicycle ID, make, model and price, and show the average price of its make as make_average_price. Keep every bicycle row and round the average to two decimal places
- `AVG(price) OVER (PARTITION BY make)` ergänzt den Durchschnitt der jeweiligen Marke in jeder Fahrradzeile, ohne Zeilen zusammenzufassen
- Format: `ROUND(CAST(AVG(wertspalte) OVER (PARTITION BY gruppenspalte) AS NUMERIC), 2) AS alias`, NULL wird beim Durchschnitt ignoriert

In [25]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price,                                                        --Alle einzelnen Fahrräder beibehalten
           ROUND(CAST(AVG(price) OVER (PARTITION BY make) AS NUMERIC), 2) AS make_average_price  --Durchschnitt innerhalb derselben Marke berechnen und runden
    FROM bicycles                                                                                --PARTITION BY bildet Berechnungsgruppen ohne Zeilen zusammenzufassen
    ORDER BY make, bicycleid;                                                                    --Fahrräder nach Marke und ID anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price,make_average_price
0,C0F02RXZ69,Bianchi,Infinito,3114.19,4556.95
1,C0J493ZL9Y,Bianchi,Aria,4358.55,4556.95
2,C0PWXC94RA,Bianchi,Aria,7174.81,4556.95
3,C0TABSD96T,Bianchi,Oltre,4007.85,4556.95
4,C0VVD5LONX,Bianchi,Oltre,7549.23,4556.95
...,...,...,...,...,...
1995,CYUXH58HTW,Trek,Domane,7705.89,4417.79
1996,CYX3L7FAJM,Trek,Emonda,2647.07,4417.79
1997,CZ1LGK51DB,Trek,Marlin,6396.66,4417.79
1998,CZ7Y3CUMRY,Trek,Marlin,3970.60,4417.79


### ROW_NUMBER — Number all bicycles with a known price from most to least expensive. Display bicycleid, make, model, price and row_number. Use bicycle ID in ascending order to break price ties
- `ROW_NUMBER()` vergibt eindeutige fortlaufende Nummern ab 1, teuerste Fahrräder zuerst und bei Preisgleichheit nach Fahrrad-ID
- Format: `ROW_NUMBER() OVER (ORDER BY price DESC, bicycleid ASC) AS row_number`, `WHERE price IS NOT NULL` schliesst fehlende Preise aus

In [26]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price,                                       --Fahrraddaten auswählen
           ROW_NUMBER() OVER (ORDER BY price DESC, bicycleid ASC) AS row_number --Eindeutige fortlaufende Nummer ab 1 vergeben; ID löst Preisgleichstände auf
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE price IS NOT NULL                                                     --Nur Fahrräder mit bekanntem Preis nummerieren
    ORDER BY row_number;                                                        --Ergebnismenge ausdrücklich nach der berechneten Nummer sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price,row_number
0,C187CGZK39,Trek,Marlin,7995.90,1
1,CENB0TLQKE,Bianchi,Oltre,7989.74,2
2,CJSLUUOM7Z,Santa Cruz,Hightower,7984.31,3
3,CLQCGBXCAG,Merida,Scultura,7972.88,4
4,CTVKH7S3PZ,Trek,Marlin,7972.58,5
...,...,...,...,...,...
1995,C7ZPXA6EXM,Cannondale,SuperSix,822.42,1996
1996,CNX3HEE5GS,Merida,Big Nine,813.47,1997
1997,CZVM7LWXRP,Specialized,Rockhopper,809.00,1998
1998,CCKYJCLZKE,Santa Cruz,Bronson,802.81,1999


### RANK — Rank all bicycles with a known price from most to least expensive. Display bicycleid, make, model, price and price_rank. Equal prices must receive the same rank, with gaps after ties
- `RANK()` gibt gleichen Preisen denselben Rang, danach entstehen Ranglücken, bspw. 1, 1, 3
- Format: `RANK() OVER (ORDER BY price DESC) AS price_rank`, keine eindeutige ID innerhalb von OVER ergänzen, sonst erhalten gleiche Preise unterschiedliche Ränge

In [27]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price,                                       --Fahrraddaten auswählen
           RANK() OVER (ORDER BY price DESC) AS price_rank                      --Gleiche Preise erhalten denselben Rang; danach entstehen Ranglücken
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE price IS NOT NULL                                                     --Fehlende Preise ausschliessen
    ORDER BY price_rank, bicycleid;                                             --Rangfolge anzeigen, ID sortiert Gleichstände, verändert aber nicht den Rang
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price,price_rank
0,C187CGZK39,Trek,Marlin,7995.90,1
1,CENB0TLQKE,Bianchi,Oltre,7989.74,2
2,CJSLUUOM7Z,Santa Cruz,Hightower,7984.31,3
3,CLQCGBXCAG,Merida,Scultura,7972.88,4
4,CTVKH7S3PZ,Trek,Marlin,7972.58,5
...,...,...,...,...,...
1995,C7ZPXA6EXM,Cannondale,SuperSix,822.42,1996
1996,CNX3HEE5GS,Merida,Big Nine,813.47,1997
1997,CZVM7LWXRP,Specialized,Rockhopper,809.00,1998
1998,CCKYJCLZKE,Santa Cruz,Bronson,802.81,1999


### UNION — Combine the bicycle IDs, makes and models of Bianchi bicycles with those of bicycles priced above 3000. Remove duplicate result rows
- `UNION` führt beide Ergebnisse untereinander zusammen und entfernt doppelte vollständige Ergebniszeilen, ein teures Bianchi-Fahrrad erscheint nur einmal
- Format: `SELECT spalte1, spalte2 FROM tabelle1 UNION SELECT spalte1, spalte2 FROM tabelle2`, beide Abfragen benötigen gleich viele Spalten mit kompatiblen Datentypen in derselben Reihenfolge

In [28]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model                                               --Erste Ergebnismenge: Bianchi-Fahrräder
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE make = 'Bianchi'                                                      --Auf Marke Bianchi filtern
    UNION                                                                       --Ergebnismengen untereinander verbinden; identische vollständige Zeilen entfernen
    SELECT bicycleid, make, model                                               --Zweite Abfrage braucht gleich viele Spalten mit kompatiblen Datentypen
    FROM bicycles                                                               --Erneut aus der Fahrradtabelle lesen
    WHERE price > 3000                                                          --Fahrräder über 3000 auswählen
    ORDER BY bicycleid;                                                         --Gesamtes vereinigtes Ergebnis sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model
0,C02MOC0YLP,Cube,Reaction
1,C03X2W8T7W,Cannondale,Synapse
2,C04CCUTU6Z,Scott,Spark
3,C04U0GF99X,Merida,Big Nine
4,C04UVK6635,Specialized,Roubaix
...,...,...,...
1436,CZWIKWF30V,Cervelo,Caledonia
1437,CZY9JAU92V,Cervelo,Soloist
1438,CZYTYT5C77,Scott,Addict
1439,CZZEBAGXVC,Scott,Aspect


### UNION ALL — Combine the bicycle IDs, makes and models of Bianchi bicycles with those of bicycles priced above 3000. Keep duplicate result rows when a bicycle meets both conditions
- `UNION ALL` hängt beide Ergebnisse untereinander an und behält Duplikate, ein Bianchi-Fahrrad über 3000 erscheint zweimal
- Format: `SELECT spalte1, spalte2 FROM tabelle1 UNION ALL SELECT spalte1, spalte2 FROM tabelle2`, beide Abfragen benötigen gleich viele Spalten mit kompatiblen Datentypen in derselben Reihenfolge

In [29]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model                                               --Erste Ergebnismenge auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE make = 'Bianchi'                                                      --Bianchi-Fahrräder auswählen
    UNION ALL                                                                   --Ergebnisse anhängen, ohne doppelte Zeilen zu entfernen
    SELECT bicycleid, make, model                                               --Spaltenanzahl, Reihenfolge und Datentypen müssen zusammenpassen
    FROM bicycles                                                               --Zweite Ergebnismenge aus derselben Tabelle auswählen
    WHERE price > 3000                                                          --Fahrräder über 3000 auswählen; teure Bianchi-Fahrräder erscheinen zweimal
    ORDER BY bicycleid;                                                         --Gesamtes Ergebnis nach Fahrrad-ID sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model
0,C02MOC0YLP,Cube,Reaction
1,C03X2W8T7W,Cannondale,Synapse
2,C04CCUTU6Z,Scott,Spark
3,C04U0GF99X,Merida,Big Nine
4,C04UVK6635,Specialized,Roubaix
...,...,...,...
1580,CZWIKWF30V,Cervelo,Caledonia
1581,CZY9JAU92V,Cervelo,Soloist
1582,CZYTYT5C77,Scott,Addict
1583,CZZEBAGXVC,Scott,Aspect


## PostgreSQl Regex

### Regex in SELECT – Treffer als TRUE oder FALSE anzeigen
- Ein Regex-Ausdruck im SELECT ergänzt eine Ergebnisspalte mit TRUE bei Treffer und FALSE ohne Treffer, bei NULL entsteht NULL
- Alle Fahrradzeilen bleiben erhalten, erst eine Regex-Bedingung in WHERE filtert Zeilen
- Format: `SELECT spalte ~ 'muster' AS alias FROM tabelle`

In [30]:
engine = create_engine(db_connection_url)                                       #Engine für den Datenbankzugriff erstellen

df_spickap01 = pd.read_sql_query(r"""
    SELECT bicycleid, make, model, framenr,                         --Originalwerte neben den Prüfergebnissen anzeigen
           make ~ 'Trek' AS matches_case,                           --TRUE wenn die Marke Trek enthält mit exakt dieser Schreibweise
           make ~* 'trek' AS matches_ignore_case,                   --TRUE wenn die Marke trek enthält unabhängig von Gross- und Kleinschreibung
           make !~ 'Trek' AS excludes_case,                         --TRUE wenn die Marke kein Trek mit exakt dieser Schreibweise enthält
           make !~* 'trek' AS excludes_ignore_case,                 --TRUE wenn die Marke kein trek enthält unabhängig von Gross- und Kleinschreibung
           model ~ '^D' AS starts_with_d,                           --TRUE wenn das Modell mit grossem D beginnt
           model ~ 'e$' AS ends_with_e,                             --TRUE wenn das Modell mit kleinem e endet
           make ~ '^Trek$' AS exactly_trek,                         --TRUE wenn die gesamte Marke exakt Trek ist
           model ~ 'a.e' AS contains_a_any_e,                       --TRUE wenn das Modell a und e mit genau einem Zeichen dazwischen enthält
           model ~* '[aeiou]' AS contains_vowel,                    --TRUE wenn das Modell mind. einen der Vokale a e i o u enthält inkl Grossbuchstaben
           framenr ~ '[0-9]' AS contains_digit,                     --TRUE wenn die Rahmennummer mind. eine Ziffer enthält
           framenr ~ '[^0-9]' AS contains_non_digit,                --TRUE wenn die Rahmennummer mind. ein Zeichen enthält das keine Ziffer ist
           framenr ~ '\d' AS digit_shortcut,                        --TRUE wenn die Rahmennummer mind. eine Ziffer enthält
           model ~ '\s' AS contains_whitespace,                     --TRUE wenn das Modell mind. ein Leerraumzeichen enthält bspw ein Leerzeichen
           model ~ '^\w+$' AS only_word_characters,                 --TRUE wenn das gesamte Modell nur aus Buchstaben Ziffern oder Unterstrichen besteht und nicht leer ist
           model ~ '^D.*e$' AS starts_d_ends_e,                     --TRUE wenn das Modell mit D beginnt und mit e endet dazwischen sind beliebig viele Zeichen erlaubt
           framenr ~ '^[A-Z]+$' AS only_uppercase_letters,          --TRUE wenn die gesamte Rahmennummer nur aus Grossbuchstaben A bis Z besteht und nicht leer ist
           model ~ '^Big\s?Nine$' AS optional_space,                --TRUE bei exakt BigNine oder Big und Nine mit genau einem Leerraumzeichen dazwischen
           bicycleid ~ '^[A-Z0-9]{10}$' AS id_has_ten_chars,        --TRUE wenn die gesamte Fahrrad-ID genau zehn Zeichen aus A bis Z oder 0 bis 9 enthält
           framenr ~ '\d{2,4}' AS two_to_four_digits,               --TRUE wenn die Rahmennummer mind. zwei Ziffern direkt hintereinander enthält auch innerhalb längerer Ziffernfolgen
           framenr ~ '\d{3,}' AS at_least_three_digits,             --TRUE wenn die Rahmennummer mind. drei Ziffern direkt hintereinander enthält
           make ~ '^(Trek|Merida)$' AS selected_brands,             --TRUE wenn die gesamte Marke exakt Trek oder Merida ist
           model ~ '\.' AS contains_literal_dot,                    --TRUE wenn das Modell einen tatsächlichen Punkt enthält
           model ~ '\mNine\M' AS contains_word_nine                 --TRUE wenn Nine als vollständiges Wort vorkommt bspw Big Nine aber nicht BigNine
    FROM bicycles                                                   --Alle Fahrräder behalten bei fehlendem Ausgangswert ergibt die jeweilige Prüfung NULL
    ORDER BY bicycleid;                                             --Ergebnisse nach Fahrrad-ID sortieren
""", con=engine)                                                                #SQL ausführen und TRUE FALSE oder NULL je Prüfung als DataFrame laden

engine.dispose()                                                                #Vom Verbindungspool gehaltene Verbindungen freigeben
display(df_spickap01)                        #Originalwerte und Prüfergebnisse anzeigen

,bicycleid,make,model,framenr,matches_case,matches_ignore_case,excludes_case,excludes_ignore_case,starts_with_d,ends_with_e,...,only_word_characters,starts_d_ends_e,only_uppercase_letters,optional_space,id_has_ten_chars,two_to_four_digits,at_least_three_digits,selected_brands,contains_literal_dot,contains_word_nine
0,C02MOC0YLP,Cube,Reaction,XMD5040,False,False,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
1,C037S02BZX,Cannondale,SuperSix,JRN 233,False,False,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
2,C03X2W8T7W,Cannondale,Synapse,950 OPV,False,False,True,True,False,True,...,True,False,False,False,True,True,True,False,False,False
3,C04CCUTU6Z,Scott,Spark,72Y TB8,False,False,True,True,False,False,...,True,False,False,False,True,True,False,False,False,False
4,C04U0GF99X,Merida,Big Nine,0019,False,False,True,True,False,True,...,False,False,False,True,True,True,True,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,CZXS1CHZTV,Cervelo,Caledonia,DRU-3873,False,False,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
1996,CZY9JAU92V,Cervelo,Soloist,456 4340,False,False,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
1997,CZYTYT5C77,Scott,Addict,386M,False,False,True,True,False,False,...,True,False,False,False,True,True,True,False,False,False
1998,CZZEBAGXVC,Scott,Aspect,NG2 T9O,False,False,True,True,False,False,...,True,False,False,False,True,False,False,False,False,False


### Regex als WHERE-Bedingung
- Eine Regex-Bedingung in WHERE behält nur Zeilen, deren Prüfung TRUE ergibt, FALSE und NULL werden ausgeschlossen
- Format: `SELECT spalten FROM tabelle WHERE spalte ~ 'muster'`, `~` berücksichtigt Gross- und Kleinschreibung und `~*` ignoriert sie

In [ ]:
engine = create_engine(db_connection_url)                       #Engine für den Datenbankzugriff erstellen

#Zusätzliche Regex-Beispiele sind auskommentiert da sie teilweise anderen Bedingungen widersprechen
#Zum Verwenden gewünschte Zeile aktivieren und nicht benötigte Bedingungen auskommentieren

df_spickap01 = pd.read_sql_query(r"""
    SELECT bicycleid, make, model, framenr                      --Ausgewählte Fahrradspalten anzeigen
    FROM bicycles                                               --Fahrradtabelle verwenden
    WHERE make ~ '^(Trek|Merida)$'                              --Nur Marken die exakt Trek oder Merida entsprechen
      AND make ~* '^(trek|merida)$'                             --Nur Trek oder Merida unabhängig von Gross-/Kleinschreibung hier bereits erfüllt
      AND make !~ '^Scott$'                                     --Marke darf nicht exakt Scott sein hier bereits ausgeschlossen
      AND make !~* '^giant$'                                    --Marke darf nicht Giant sein unabhängig von Gross-/Kleinschreibung hier bereits ausgeschlossen
      --AND make ~ '^[BT]'                                      --Marke beginnt mit grossem B oder T bspw. Bianchi oder Trek

      AND model ~ '^[A-Z].*$'                                   --Modell beginnt mit einem Grossbuchstaben A bis Z danach beliebig viele Zeichen auch keines
      AND model ~ '[aeiou]'                                     --Modell enthält mind. einen kleinen Vokal a e i o u
      AND model ~ '\w+'                                         --Modell enthält mind. ein Wortzeichen bspw. Buchstabe Ziffer oder Unterstrich hier bereits erfüllt
      AND model !~ '\d'                                         --Modell darf keine Ziffer enthalten
      AND model !~ '\.'                                         --Modell darf keinen tatsächlichen Punkt enthalten
      --AND model ~ '^Dom.*e$'                                  --Modell beginnt mit Dom und endet mit e dazwischen beliebig viele Zeichen auch keines
      --AND model ~ '^B.+i$'                                    --Modell beginnt mit B und endet mit i dazwischen mind. ein beliebiges Zeichen
      --AND model ~ '\D'                                        --Modell enthält mind. ein Zeichen das keine Ziffer ist
      --AND model ~ '\S'                                        --Modell enthält mind. ein Zeichen das kein Leerraum ist
      --AND model ~ '\W'                                        --Modell enthält mind. ein Nichtwortzeichen bspw. Leerzeichen Punkt oder Bindestrich

      AND CAST(bicycleid AS TEXT) ~ '^[A-Z0-9]{10}$'            --Gesamte Fahrrad-ID hat genau zehn Zeichen aus A bis Z oder 0 bis 9 CAST erlaubt auch numerische Ausgangsspalten

      AND framenr ~ '[^0-9]'                                    --Rahmennummer enthält mind. ein Zeichen das keine Ziffer ist
      AND framenr ~ '\d{2,4}'                                   --Rahmennummer enthält zwei bis vier direkt aufeinanderfolgende Ziffern auch als Teil einer längeren Ziffernfolge
      --AND framenr ~ '\d{3,}'                                  --Rahmennummer enthält mind. drei Ziffern direkt hintereinander würde bspw. A15 0LX ausschliessen
      --AND framenr ~ '^[^0-9]+$'                               --Gesamte Rahmennummer enthält keine Ziffer und ist nicht leer widerspricht den Ziffernbedingungen

      AND (
          framenr ~ '^[A-Z]{3}-?\d+$'                           --Drei Grossbuchstaben optionaler Bindestrich und mind. eine Ziffer bspw. ABC123 oder ABC-123
          OR framenr ~ '^[A-Z]\d+\s\d[A-Z]{2}$'                 --Grossbuchstabe Ziffernfolge ein Leerraumzeichen eine Ziffer und zwei Grossbuchstaben bspw. A15 0LX
      )                                                         --Eines der Formate muss passen zusätzlich gelten alle aktiven AND-Bedingungen
    ORDER BY bicycleid;                                         --Treffer nach Fahrrad-ID sortieren
""", con=engine)                                                #SQL ausführen und gefilterte Zeilen laden

engine.dispose()                                                #Vom Verbindungspool gehaltene Verbindungen freigeben
display(df_spickap01)                                           #Gefilterte Fahrräder anzeigen

,bicycleid,make,model,framenr
0,C06Z017RMC,Merida,Scultura,NEH-2263
1,C1HFOMEBDK,Merida,Scultura,IOH-141
2,C1U71ESAXN,Trek,Emonda,HRN-0839
3,C2Q38KWLVE,Trek,Emonda,SLF-4020
4,C33QM7379T,Merida,Scultura,N76 6YA
...,...,...,...,...
59,CW6FLJVQFS,Merida,Reacto,Y67 9XX
60,CXT6RTRLPI,Trek,Domane,BMG-7319
61,CYWCX4O6DY,Merida,Big Nine,POP3628
62,CZ1LGK51DB,Trek,Marlin,WEK6974


### Regex ~ '^B' — Retrieve the bicycle ID, make, model and price of all bicycles whose make starts with an uppercase B
- `make ~ '^B'` wählt Marken mit grossem B am Anfang aus, `^` steht für den Textanfang
- Format: `WHERE spalte ~ '^muster'`, `~` berücksichtigt Gross- und Kleinschreibung

In [32]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE make ~ '^B';                                                          --^ markiert den Textanfang; ~ berücksichtigt Gross-/Kleinschreibung
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Regex ~* — Retrieve bicycle ID, make, model and price for bicycles whose make starts with B or b. Use a case-insensitive regular expression
- `make ~* '^b'` wählt Marken mit B oder b am Anfang aus, `~*` ignoriert Gross- und Kleinschreibung
- Format: `WHERE spalte ~* '^muster'`, `^` steht für den Textanfang

In [33]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE make ~* '^b';                                                         --~* ignoriert Gross-/Kleinschreibung; ^ verlangt den Textanfang
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Regex $ — Retrieve bicycle ID, make, model and price for bicycles whose make ends with "chi". Use a regular expression
- `make ~ 'chi$'` wählt Marken mit exakt kleingeschriebenem chi am Ende aus, `$` steht für das Textende
- Format: `WHERE spalte ~ 'muster$'`, mit `~*` wird Gross- und Kleinschreibung ignoriert

In [34]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE make ~ 'chi$';                                                        --$ verlangt das Textende; chi muss am Ende der Marke stehen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Regex ^...$ — Retrieve bicycle ID, make, model and price for bicycles whose make is exactly "Bianchi". Use a regular expression
- `make ~ '^Bianchi$'` verlangt exakt Bianchi als gesamten Wert, zusätzliche Zeichen davor oder danach sind nicht erlaubt
- Format: `WHERE spalte ~ '^muster$'`, `^` und `$` begrenzen das Muster auf den gesamten Text

In [35]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE make ~ '^Bianchi$';                                                   --^ und $ begrenzen den gesamten Text; keine zusätzlichen Zeichen erlaubt
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Regex . — Retrieve bicycle ID, make, model and price for bicycles whose make starts with B, ends with i and has exactly five characters between them
- `make ~ '^B.....i$'` verlangt B am Anfang, i am Ende und genau fünf beliebige Zeichen dazwischen, insgesamt sieben Zeichen
- Format: `WHERE spalte ~ '^B.{5}i$'` ist gleichbedeutend, `.` steht für genau ein beliebiges Zeichen

In [36]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE make ~ '^B.....i$';                                                   --Jeder Punkt steht für genau ein Zeichen; fünf Punkte zwischen B und i
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C2ZEK0YPD9,Bianchi,Oltre,7382.81
1,CTFUCPJ6CQ,Bianchi,Aria,7050.68
2,CCP6K2ZAAU,Bianchi,Aria,3965.39
3,CQ61HMS317,Bianchi,Aria,4180.60
4,CJSMB9CVGE,Bianchi,Infinito,1788.59
...,...,...,...,...
186,C6FYCWDJKF,Bianchi,Infinito,6983.40
187,CVG2ERXCAM,Bianchi,Oltre,3068.96
188,CB4PK8PLSN,Bianchi,Oltre,7765.04
189,CFFTG1MNW0,Bianchi,Aria,5939.92


### Regex character class — Retrieve bicycle ID, make, model and price for bicycles whose make starts with an uppercase B or T. Use a character class
- `make ~ '^[BT]'` wählt Marken mit grossem B oder T am Anfang aus, `[BT]` steht für genau eines dieser Zeichen
- Format: `WHERE spalte ~ '^[zeichen]'`, ohne `^` genügt ein passendes Zeichen an beliebiger Stelle

In [37]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE make ~ '^[BT]';                                                       --[BT] erlaubt genau ein Zeichen: B oder T; ^ verlangt den Textanfang
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C2UQZP803C,Trek,Marlin,5038.99
2,C2ZEK0YPD9,Bianchi,Oltre,7382.81
3,CE5XORIKH7,Trek,Emonda,7600.96
4,C4ZC4AYR1E,Trek,Emonda,2198.88
...,...,...,...,...
371,CFFTG1MNW0,Bianchi,Aria,5939.92
372,C6DHR3STN7,Trek,Marlin,6624.90
373,CLH919EIKV,Trek,Emonda,2376.02
374,CP5CTME2ZD,Trek,Marlin,6176.55


### Regex [0-9] — Retrieve bicycle ID, make, model and price for bicycles whose model name contains at least one digit
- `model ~ '[0-9]'` wählt Modellnamen mit mindestens einer Ziffer an beliebiger Stelle aus, bspw. Bike 2000
- Format: `WHERE spalte ~ '[0-9]'`, weitere Buchstaben oder Sonderzeichen sind erlaubt

In [38]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE model ~ '[0-9]';                                                      --Eine Ziffer an beliebiger Stelle genügt; ohne Anker wird irgendwo gesucht
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CN55MXKWDD,Santa Cruz,5010,7590.43
1,CRRQDDQ6HZ,Cervelo,R5,3044.49
2,C1RTS221QB,Santa Cruz,5010,2543.34
3,CXKGSTPPRY,Santa Cruz,5010,3557.11
4,C77RP25IYD,Santa Cruz,5010,4439.59
...,...,...,...,...
120,C4NID31AE4,Santa Cruz,5010,2759.39
121,CRS99LKJTV,Santa Cruz,5010,5107.08
122,CW30JYTH5F,Santa Cruz,5010,7858.48
123,CXSPK20YP0,Cervelo,R5,3719.35


### Regex [^0-9] — Retrieve bicycle ID, make, model and price for bicycles whose model name contains at least one character that is not a digit
- `model ~ '[^0-9]'` wählt Modellnamen mit mind. einem Nicht-Ziffernzeichen aus, bspw. Buchstabe, Leerzeichen oder Punkt
- Format: `WHERE spalte ~ '[^0-9]'`, `^` innerhalb der Zeichenklasse bedeutet Ausschluss. Bike 2000 passt, 2000 nicht

In [39]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE model ~ '[^0-9]';                                                     --^ innerhalb der Zeichenklasse bedeutet Ausschluss, auch Leerzeichen zählen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1913,CLH919EIKV,Trek,Emonda,2376.02
1914,CP3F48BE9Y,Merida,Scultura,2833.36
1915,CHBXQ3SSNU,Merida,Scultura,1629.91
1916,CP5CTME2ZD,Trek,Marlin,6176.55


### Regex * — Retrieve bicycle ID, make, model and price for bicycles whose model name starts with "Dom" and ends with "e", allowing any number of characters in between
- `model ~ '^Dom.*e$'` verlangt Dom am Anfang und e am Ende, dazwischen sind beliebig viele Zeichen erlaubt inkl. keinem Zeichen, bspw. Dome oder Domane
- Format: `WHERE spalte ~ '^anfang.*ende$'`, `*` wiederholt das vorherige Element nullmal oder beliebig oft und `.*` steht damit für beliebig viele Zeichen

In [40]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE model ~ '^Dom.*e$';                                                   --.* erlaubt null oder beliebig viele Zeichen zwischen Dom und e
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,CTLAL7CVQU,Trek,Domane,2390.08
2,CIBOKX0ZK2,Trek,Domane,5390.75
3,CGQLKFPRL6,Trek,Domane,6790.75
4,CTM3DK199V,Trek,Domane,5486.52
5,CJDBDJDR48,Trek,Domane,7673.22
6,C7TLRFVXV2,Trek,Domane,1205.41
7,CFTMLVG7JJ,Trek,Domane,5742.05
8,CYEUN8XEM0,Trek,Domane,1091.15
9,C32MK7YOH3,Trek,Domane,1074.92


### Regex + — Retrieve bicycle ID, make, model and price for bicycles whose model name consists entirely of one or more digits
- `model ~ '^[0-9]+$'` verlangt ausschliesslich Ziffern und mind. eine davon. 7 und 2000 passen, Bike 2000 nicht
- Format: `WHERE spalte ~ '^[0-9]+$'`, `+` wiederholt das vorherige Element mind. einmal und `^` sowie `$` begrenzen den gesamten Text

In [41]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE model ~ '^[0-9]+$';                                                   --+ verlangt mind. eine Ziffer; ^ und $ schliessen andere Zeichen aus
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CN55MXKWDD,Santa Cruz,5010,7590.43
1,C1RTS221QB,Santa Cruz,5010,2543.34
2,CXKGSTPPRY,Santa Cruz,5010,3557.11
3,C77RP25IYD,Santa Cruz,5010,4439.59
4,CS75T5RDH7,Santa Cruz,5010,7894.35
...,...,...,...,...
77,CL85C79DVD,Santa Cruz,5010,6215.09
78,CU789362X0,Santa Cruz,5010,1283.59
79,C4NID31AE4,Santa Cruz,5010,2759.39
80,CRS99LKJTV,Santa Cruz,5010,5107.08


### Regex ? — Retrieve bicycle ID, make, model and price for bicycles whose model name is exactly "Domane" or "Domanes". Use an optional character
- `model ~ '^Domanes?$'` findet exakt Domane oder Domanes, nur das letzte s ist optional
- Format: `WHERE spalte ~ '^textzeichen?$'`, `?` erlaubt das unmittelbar vorherige Element nullmal oder einmal

In [42]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE model ~ '^Domanes?$';                                                 --? macht das unmittelbar vorhergehende s optional; Domane und Domanes passen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,CTLAL7CVQU,Trek,Domane,2390.08
2,CIBOKX0ZK2,Trek,Domane,5390.75
3,CGQLKFPRL6,Trek,Domane,6790.75
4,CTM3DK199V,Trek,Domane,5486.52
5,CJDBDJDR48,Trek,Domane,7673.22
6,C7TLRFVXV2,Trek,Domane,1205.41
7,CFTMLVG7JJ,Trek,Domane,5742.05
8,CYEUN8XEM0,Trek,Domane,1091.15
9,C32MK7YOH3,Trek,Domane,1074.92


### Regex {n} — Retrieve bicycle ID, make, model and price for bicycles whose ID starts with C followed by exactly nine uppercase letters or digits
- `bicycleid ~ '^C[A-Z0-9]{9}$'` verlangt C gefolgt von genau neun Grossbuchstaben oder Ziffern, insgesamt zehn Zeichen
- Format: `WHERE spalte ~ '^C[A-Z0-9]{n}$'`, `{n}` wiederholt das unmittelbar vorherige Element genau n-mal

In [43]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE bicycleid ~ '^C[A-Z0-9]{9}$';                                         --{9} verlangt genau neun Zeichen aus der Klasse nach dem führenden C
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C875HZ1AG6,Merida,Reacto,5529.38
2,CDONEWSWFP,Merida,Big Nine,1518.33
3,C2UQZP803C,Trek,Marlin,5038.99
4,C2WFIK6WEP,Merida,Scultura,1548.19
...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2376.02
1996,CP3F48BE9Y,Merida,Scultura,2833.36
1997,CHBXQ3SSNU,Merida,Scultura,1629.91
1998,CP5CTME2ZD,Trek,Marlin,6176.55


### Regex {n,m} — Retrieve bicycle ID, make, model and price for bicycles whose model name consists entirely of two to four digits
- `model ~ '^[0-9]{2,4}$'` verlangt ausschliesslich zwei bis vier Ziffern. 12, 123 und 1234 passen, 1 und 12345 nicht
- Format: `WHERE spalte ~ '^[0-9]{n,m}$'`, `{n,m}` wiederholt das vorherige Element mind. n-mal und höchstens m-mal

In [44]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE model ~ '^[0-9]{2,4}$';                                               --{2,4} erlaubt mind. zwei und höchstens vier Ziffern im gesamten Text
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,CN55MXKWDD,Santa Cruz,5010,7590.43
1,C1RTS221QB,Santa Cruz,5010,2543.34
2,CXKGSTPPRY,Santa Cruz,5010,3557.11
3,C77RP25IYD,Santa Cruz,5010,4439.59
4,CS75T5RDH7,Santa Cruz,5010,7894.35
...,...,...,...,...
77,CL85C79DVD,Santa Cruz,5010,6215.09
78,CU789362X0,Santa Cruz,5010,1283.59
79,C4NID31AE4,Santa Cruz,5010,2759.39
80,CRS99LKJTV,Santa Cruz,5010,5107.08


### Regex alternatives — Retrieve bicycle ID, make, model and price for bicycles whose make is exactly Bianchi or Trek. Use grouped alternatives
- `make ~ '^(Bianchi|Trek)$'` findet exakt Bianchi oder Trek, zusätzliche Zeichen sind nicht erlaubt
- Format: `WHERE spalte ~ '^(alternative1|alternative2)$'`, `|` bedeutet oder und die Klammern lassen beide Alternativen zwischen Textanfang und Textende gelten

In [45]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE make ~ '^(Bianchi|Trek)$';                                            --Klammern gruppieren Alternativen. | bedeutet ODER, beide Anker gelten für beide Marken
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C7ZGWFLBUV,Trek,Domane,5066.94
1,C2UQZP803C,Trek,Marlin,5038.99
2,C2ZEK0YPD9,Bianchi,Oltre,7382.81
3,CE5XORIKH7,Trek,Emonda,7600.96
4,C4ZC4AYR1E,Trek,Emonda,2198.88
...,...,...,...,...
371,CFFTG1MNW0,Bianchi,Aria,5939.92
372,C6DHR3STN7,Trek,Marlin,6624.90
373,CLH919EIKV,Trek,Emonda,2376.02
374,CP5CTME2ZD,Trek,Marlin,6176.55


### Regex literal dot — Retrieve bicycle ID, make, model and price for bicycles whose model name contains a literal dot. Use a regular expression
- `model ~ '\.'` sucht einen tatsächlichen Punkt im Modellnamen, ohne Backslash steht `.` für ein beliebiges Zeichen
- Format: `WHERE spalte ~ '\.'`, alternativ `[.]` verwenden und im Python-Code `r"""..."""` nutzen, damit der Backslash unverändert weitergegeben wird

In [46]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Fahrrad-ID, Marke, Modell und Preis auswählen
    FROM bicycles                                                               --Fahrradtabelle verwenden
    WHERE model ~ '[.]';                                                        --[.] sucht einen tatsächlichen Punkt, ohne passende Modellnamen ist das Ergebnis leer
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price


### INSERT INTO — Create an empty temporary bicycle table and insert the existing sample bicycle C02MOC0YLP: Cube Reaction, year 2020, price 4823.86, status Sold. Display the inserted row
- `INSERT INTO` fügt eine Zeile in die zuvor erstellte temporäre Tabelle ein, die Reihenfolge der Werte muss zur angegebenen Spaltenreihenfolge passen
- Format: `INSERT INTO tabelle (spalte1, spalte2) VALUES (wert1, wert2)`, Texte in einfachen Anführungszeichen und Zahlen ohne Anführungszeichen

In [47]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles (                                          --Leere temporäre Übungstabelle mit bekannten Bike-Shop-Spalten erstellen
        bicycleid TEXT,                                                         --Fahrrad-ID als Text speichern
        make VARCHAR(100),                                                      --Marke als Text mit maximal 100 Zeichen speichern
        model VARCHAR(100),                                                     --Modell als Text mit maximal 100 Zeichen speichern
        year INTEGER,                                                           --Baujahr als Ganzzahl speichern
        price NUMERIC(10, 2),                                                   --Preis mit insgesamt zehn Stellen, davon zwei Nachkommastellen
        status TEXT DEFAULT 'Available'                                         --Standardstatus für ausgelassene Werte oder DEFAULT festlegen
    ) ON COMMIT DROP;                                                           --Übungstabelle am Transaktionsende automatisch entfernen
    """))

    connection.execute(text("""
    INSERT INTO spick_bicycles (bicycleid, make, model, year, price, status)    --Zielspalten ausdrücklich in der Reihenfolge der Werte angeben
    VALUES ('C02MOC0YLP', 'Cube', 'Reaction', 2020, 4823.86, 'Sold');           --Einen geprüften Datensatz aus der Quelldatei einfügen, Texte in einfachen Anführungszeichen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                    --Inhalt der Übungstabelle nach der Änderung anzeigen
    FROM spick_bicycles                                                         --Ausschliesslich die temporäre Kopie abfragen
    ORDER BY bicycleid;                                                         --Ergebnis nach Fahrrad-ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,year,price,status
0,C02MOC0YLP,Cube,Reaction,2020,4823.86,Sold


### INSERT multiple rows — Create an empty temporary bicycle table and insert both existing samples in one INSERT: C02MOC0YLP (Cube, Reaction, 2020, 4823.86, Sold) and C037S02BZX (Cannondale, SuperSix, 2023, 2424.03, Sold). Display both rows
- Ein INSERT kann mehrere Zeilen einfügen, jedes Klammerpaar nach VALUES beschreibt eine Zeile und wird durch ein Komma vom nächsten getrennt
- Format: `INSERT INTO tabelle (spalte1, spalte2) VALUES (wert1, wert2), (wert3, wert4)`, alle Zeilen müssen zur angegebenen Spaltenreihenfolge passen

In [48]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles (                                          --Leere temporäre Übungstabelle mit bekannten Bike-Shop-Spalten erstellen
        bicycleid TEXT,                                                         --Fahrrad-ID als Text speichern
        make VARCHAR(100),                                                      --Marke als Text mit maximal 100 Zeichen speichern
        model VARCHAR(100),                                                     --Modell als Text mit maximal 100 Zeichen speichern
        year INTEGER,                                                           --Baujahr als Ganzzahl speichern
        price NUMERIC(10, 2),                                                   --Preis mit insgesamt zehn Stellen, davon zwei Nachkommastellen
        status TEXT DEFAULT 'Available'                                         --Standardstatus für ausgelassene Werte oder DEFAULT festlegen
    ) ON COMMIT DROP;                                                           --Übungstabelle am Transaktionsende automatisch entfernen
    """))

    connection.execute(text("""
    INSERT INTO spick_bicycles (bicycleid, make, model, year, price, status)    --Spaltenreihenfolge für beide neuen Zeilen festlegen
    VALUES                                                                      --Mehrere Zeilen mit einer INSERT-Anweisung einfügen
        ('C02MOC0YLP', 'Cube', 'Reaction', 2020, 4823.86, 'Sold'),              --Ersten geprüften Datensatz einfügen; Komma trennt die Zeilen
        ('C037S02BZX', 'Cannondale', 'SuperSix', 2023, 2424.03, 'Sold');        --Zweiten geprüften Datensatz einfügen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                    --Inhalt der Übungstabelle nach der Änderung anzeigen
    FROM spick_bicycles                                                         --Ausschliesslich die temporäre Kopie abfragen
    ORDER BY bicycleid;                                                         --Ergebnis nach Fahrrad-ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,year,price,status
0,C02MOC0YLP,Cube,Reaction,2020,4823.86,Sold
1,C037S02BZX,Cannondale,SuperSix,2023,2424.03,Sold


### INSERT INTO SELECT — Create an empty temporary bicycle table and copy all Bianchi bicycles into it using INSERT INTO SELECT. Display the copied rows
- `INSERT INTO ... SELECT` kopiert die ausgewählten Bianchi-Fahrräder in die bereits erstellte temporäre Tabelle, die Quelldaten bleiben unverändert
- Format: `INSERT INTO zieltabelle (spalte1, spalte2) SELECT spalte1, spalte2 FROM quelltabelle WHERE bedingung`, Spaltenreihenfolge und Datentypen müssen zusammenpassen

In [49]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles (                                          --Leere temporäre Übungstabelle mit bekannten Bike-Shop-Spalten erstellen
        bicycleid TEXT,                                                         --Fahrrad-ID als Text speichern
        make VARCHAR(100),                                                      --Marke als Text mit max. 100 Zeichen speichern
        model VARCHAR(100),                                                     --Modell als Text mit max. 100 Zeichen speichern
        year INTEGER,                                                           --Baujahr als Ganzzahl speichern
        price NUMERIC(10, 2),                                                   --Preis mit insgesamt zehn Stellen, davon zwei Nachkommastellen
        status TEXT DEFAULT 'Available'                                         --Standardstatus für ausgelassene Werte oder DEFAULT festlegen
    ) ON COMMIT DROP;                                                           --Übungstabelle am Transaktionsende automatisch entfernen
    """))

    connection.execute(text("""
    INSERT INTO spick_bicycles (bicycleid, make, model, year, price, status)    --Bereits bestehende Zieltabelle und Zielspalten festlegen
    SELECT bicycleid, make, model, year, price, status                          --Quellspalten in passender Reihenfolge und mit kompatiblen Datentypen auswählen
    FROM bicycles                                                               --Bestehende Fahrraddaten als Quelle verwenden
    WHERE make = 'Bianchi';                                                     --Nur Bianchi-Fahrräder kopieren, Originaltabelle unverändert lassen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                    --Inhalt der Übungstabelle nach der Änderung anzeigen
    FROM spick_bicycles                                                         --Ausschliesslich die temporäre Kopie abfragen
    ORDER BY bicycleid;                                                         --Ergebnis nach Fahrrad-ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,year,price,status
0,C0F02RXZ69,Bianchi,Infinito,2020,3114.19,Sold
1,C0J493ZL9Y,Bianchi,Aria,2022,4358.55,Sold
2,C0PWXC94RA,Bianchi,Aria,2022,7174.81,In Maintenance
3,C0TABSD96T,Bianchi,Oltre,2023,4007.85,Sold
4,C0VVD5LONX,Bianchi,Oltre,2021,7549.23,Sold
...,...,...,...,...,...,...
186,CYODW15DMC,Bianchi,Infinito,2021,4580.56,Sold
187,CZ2M1IWTJO,Bianchi,Oltre,2020,4353.77,Sold
188,CZ3LE13DCY,Bianchi,Infinito,2022,5895.65,Sold
189,CZEFCIFQYM,Bianchi,Infinito,2022,3815.74,Available


### INSERT DEFAULT — Create an empty temporary bicycle table with default status Available. Insert C02MOC0YLP with its original make, model, year and price, but use DEFAULT for status. Display the row
- `DEFAULT` setzt den für die Spalte definierten Standardwert ein, hier Available für status
- Format: `INSERT INTO tabelle (spalte1, status) VALUES (wert1, DEFAULT)`, ohne definierten Standardwert entsteht NULL sofern die Spalte dies erlaubt

In [50]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles (                                          --Leere temporäre Übungstabelle mit bekannten Bike-Shop-Spalten erstellen
        bicycleid TEXT,                                                         --Fahrrad-ID als Text speichern
        make VARCHAR(100),                                                      --Marke als Text mit max. 100 Zeichen speichern
        model VARCHAR(100),                                                     --Modell als Text mit max. 100 Zeichen speichern
        year INTEGER,                                                           --Baujahr als Ganzzahl speichern
        price NUMERIC(10, 2),                                                   --Preis mit insgesamt zehn Stellen, davon zwei Nachkommastellen
        status TEXT DEFAULT 'Available'                                         --Standardstatus für ausgelassene Werte oder DEFAULT festlegen
    ) ON COMMIT DROP;                                                           --Übungstabelle am Transaktionsende automatisch entfernen
    """))

    connection.execute(text("""
    INSERT INTO spick_bicycles (bicycleid, make, model, year, price, status)    --Zielspalten festlegen
    VALUES ('C02MOC0YLP', 'Cube', 'Reaction', 2020, 4823.86, DEFAULT);          --Für status den definierten Standard Available verwenden statt Sold zu kopieren
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                    --Inhalt der Übungstabelle nach der Änderung anzeigen
    FROM spick_bicycles                                                         --Ausschliesslich die temporäre Kopie abfragen
    ORDER BY bicycleid;                                                         --Ergebnis nach Fahrrad-ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,year,price,status
0,C02MOC0YLP,Cube,Reaction,2020,4823.86,Available


### UPDATE SET — Create a temporary copy of bicycles. For bicycle C02MOC0YLP, change status to Available and price to 4500. Display that row
- `UPDATE ... SET` ändert Status und Preis in der temporären Kopie, `WHERE bicycleid = 'C02MOC0YLP'` begrenzt die Änderung auf dieses Fahrrad
- Format: `UPDATE tabelle SET spalte1 = wert1, spalte2 = wert2 WHERE bedingung`, ohne WHERE werden alle Zeilen geändert

In [51]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                          --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status                 --Originalspalten übernehmen
    FROM bicycles;                                                              --Originaldaten kopieren, sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    UPDATE spick_bicycles                                                       --Nur die temporäre Übungstabelle verändern
    SET status = 'Available',                                                   --Neuen Status festlegen, mehrere Zuweisungen mit Kommas trennen
        price = 4500                                                            --Neuen Preis festlegen
    WHERE bicycleid = 'C02MOC0YLP';                                             --Änderung auf die angegebene Fahrrad-ID begrenzen, ohne WHERE würden alle Zeilen geändert
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price, status                                --Geänderten Datensatz kontrollieren
    FROM spick_bicycles                                                         --Nur aus der Übungskopie lesen
    WHERE bicycleid = 'C02MOC0YLP';                                             --Das bearbeitete Fahrrad anzeigen
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price,status
0,C02MOC0YLP,Cube,Reaction,4500.0,Available


### UPDATE with calculation — Create a temporary copy of bicycles. Increase the prices of all Bianchi bicycles by 5 percent and round them to two decimal places. Display the affected bicycles
- `SET price = ROUND(CAST(price * 1.05 AS NUMERIC), 2)` erhöht die Preise um 5% und rundet auf zwei Dezimalstellen, `WHERE make = 'Bianchi'` begrenzt die Änderung auf Bianchi
- Format: `UPDATE tabelle SET spalte = berechnung WHERE bedingung`, erneutes Ausführen auf derselben Kopie erhöht die bereits geänderten Preise nochmals

In [52]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                          --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status                 --Originalspalten übernehmen
    FROM bicycles;                                                              --Originaldaten kopieren, sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    UPDATE spick_bicycles                                                       --Preise ausschliesslich in der Übungskopie ändern
    SET price = ROUND(CAST(price * 1.05 AS NUMERIC), 2)                         --Neuen Preis aus dem bisherigen Wert berechnen und runden
    WHERE make = 'Bianchi';                                                     --Nur Fahrräder der Marke Bianchi verändern
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Neue Preise kontrollieren
    FROM spick_bicycles                                                         --Temporäre Kopie abfragen
    WHERE make = 'Bianchi'                                                      --Nur betroffene Fahrräder anzeigen
    ORDER BY bicycleid;                                                         --Nach ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C0F02RXZ69,Bianchi,Infinito,3269.90
1,C0J493ZL9Y,Bianchi,Aria,4576.48
2,C0PWXC94RA,Bianchi,Aria,7533.55
3,C0TABSD96T,Bianchi,Oltre,4208.24
4,C0VVD5LONX,Bianchi,Oltre,7926.69
...,...,...,...,...
186,CYODW15DMC,Bianchi,Infinito,4809.59
187,CZ2M1IWTJO,Bianchi,Oltre,4571.46
188,CZ3LE13DCY,Bianchi,Infinito,6190.43
189,CZEFCIFQYM,Bianchi,Infinito,4006.53


### UPDATE FROM — Create a temporary copy of bicycles. Build a price_updates table with prices 10 percent above the original Bianchi prices, then transfer these prices to the copy using UPDATE FROM. Display the updated Bianchi bicycles
- `UPDATE ... FROM` übernimmt die um 10% erhöhten Preise aus price_updates in die temporäre Fahrradkopie, zugeordnet über bicycleid
- Format: `UPDATE zieltabelle AS z SET price = q.new_price FROM quelltabelle AS q WHERE z.bicycleid = q.bicycleid`, pro Zielzeile sollte höchstens eine Quellzeile passen

In [53]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                          --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status                 --Originalspalten übernehmen
    FROM bicycles;                                                              --Originaldaten kopieren, sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    CREATE TEMP TABLE price_updates ON COMMIT DROP AS                           --Temporäre Quelltabelle für neue Preise anlegen
    SELECT bicycleid,                                                           --Fahrrad-ID als Zuordnungsschlüssel übernehmen
           ROUND(CAST(price * 1.10 AS NUMERIC), 2) AS new_price                 --Neue Preise aus den Originalpreisen berechnen
    FROM bicycles                                                               --Originaldaten nur lesen
    WHERE make = 'Bianchi';                                                     --Pro Bianchi-Fahrrad genau einen Quellwert bereitstellen
    """))

    connection.execute(text("""
    UPDATE spick_bicycles AS b                                                  --Zieltabelle und Kurznamen festlegen
    SET price = p.new_price                                                     --Preis aus der Quelltabelle übernehmen
    FROM price_updates AS p                                                     --Tabelle mit neuen Preisen ergänzen
    WHERE b.bicycleid = p.bicycleid;                                            --Über ID verbinden; pro Zielzeile sollte höchstens eine Quellzeile passen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Übernommene Preise anzeigen
    FROM spick_bicycles                                                         --Temporäre Zieltabelle abfragen
    WHERE make = 'Bianchi'                                                      --Betroffene Marke auswählen
    ORDER BY bicycleid;                                                         --Nach Fahrrad-ID sortieren
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C0F02RXZ69,Bianchi,Infinito,3425.61
1,C0J493ZL9Y,Bianchi,Aria,4794.41
2,C0PWXC94RA,Bianchi,Aria,7892.29
3,C0TABSD96T,Bianchi,Oltre,4408.64
4,C0VVD5LONX,Bianchi,Oltre,8304.15
...,...,...,...,...
186,CYODW15DMC,Bianchi,Infinito,5038.62
187,CZ2M1IWTJO,Bianchi,Oltre,4789.15
188,CZ3LE13DCY,Bianchi,Infinito,6485.22
189,CZEFCIFQYM,Bianchi,Infinito,4197.31


### UPDATE NULL — Create a temporary copy of bicycles. Set the price of bicycle C02MOC0YLP to NULL and display that row
- `SET price = NULL` entfernt den Preiswert in der temporären Kopie, die Fahrradzeile bleibt erhalten und die Spalte muss NULL erlauben
- Format: `UPDATE tabelle SET spalte = NULL WHERE bedingung`, NULL ohne Anführungszeichen schreiben und anschliessend mit `IS NULL` prüfen

In [54]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                          --Temporäre Übungstabelle erstellen, beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status                 --Originalspalten übernehmen
    FROM bicycles;                                                              --Originaldaten kopieren, sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    UPDATE spick_bicycles                                                       --Nur die Übungskopie ändern
    SET price = NULL                                                            --Preis als fehlend speichern, NULL bedeutet nicht 0
    WHERE bicycleid = 'C02MOC0YLP';                                             --Änderung auf ein bestimmtes Fahrrad beschränken
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, price                                        --Datensatz mit nun fehlendem Preis anzeigen
    FROM spick_bicycles                                                         --Übungskopie abfragen
    WHERE bicycleid = 'C02MOC0YLP';                                             --Bearbeitetes Fahrrad auswählen
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C02MOC0YLP,Cube,Reaction,None


### DELETE — Create a temporary copy of bicycles. Delete bicycle C02MOC0YLP and verify that no row with this ID remains in the copy
- `DELETE FROM ... WHERE bicycleid = 'C02MOC0YLP'` löscht die Fahrradzeile aus der temporären Kopie, eine anschliessende SELECT-Abfrage für diese ID muss leer sein
- Format: `DELETE FROM tabelle WHERE bedingung`, ohne WHERE werden alle Zeilen gelöscht und die Tabellenstruktur bleibt bestehen

In [55]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                          --Temporäre Übungstabelle erstellen, beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status                 --Originalspalten übernehmen
    FROM bicycles;                                                              --Originaldaten kopieren, sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    DELETE FROM spick_bicycles                                                  --Zeile aus der Übungskopie löschen, die Tabelle bleibt bestehen
    WHERE bicycleid = 'C02MOC0YLP';                                             --Nur diese ID löschen; ohne WHERE würden alle Zeilen gelöscht
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT COUNT(*) AS remaining_rows                                           --Verbleibende Treffer zählen, erwartet wird 0
    FROM spick_bicycles                                                         --Nur die Übungstabelle prüfen
    WHERE bicycleid = 'C02MOC0YLP';                                             --Nach der gelöschten ID suchen
    """, con=connection)

engine.dispose()
display(df_spickap01)

,remaining_rows
0,0


### CREATE TABLE — Create an empty temporary table named spick_bicycles with bicycleid as text, make and model as VARCHAR(100), year as INTEGER, price as NUMERIC(10,2) and status defaulting to Available. Display its columns
- `CREATE TEMP TABLE` erstellt eine leere Tabelle für die aktuelle Datenbanksitzung, Spaltennamen und Datentypen werden dabei festgelegt
- Format: `CREATE TEMP TABLE tabelle (spalte1 datentyp, status TEXT DEFAULT 'Available')`, `NUMERIC(10,2)` erlaubt insgesamt zehn Stellen inkl. zwei Nachkommastellen

In [56]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles (                                          --Leere temporäre Übungstabelle mit bekannten Bike-Shop-Spalten erstellen
        bicycleid TEXT,                                                         --Fahrrad-ID als Text speichern
        make VARCHAR(100),                                                      --Marke als Text mit maximal 100 Zeichen speichern
        model VARCHAR(100),                                                     --Modell als Text mit maximal 100 Zeichen speichern
        year INTEGER,                                                           --Baujahr als Ganzzahl speichern
        price NUMERIC(10, 2),                                                   --Preis mit insgesamt zehn Stellen, davon zwei Nachkommastellen
        status TEXT DEFAULT 'Available'                                         --Standardstatus für ausgelassene Werte oder DEFAULT festlegen
    ) ON COMMIT DROP;                                                           --Übungstabelle am Transaktionsende automatisch entfernen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                    --Spalten der neu erstellten Tabelle anzeigen
    FROM spick_bicycles;                                                        --Leere Tabelle liefert Spaltennamen, aber noch keine Datenzeilen
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,year,price,status


### SELECT INTO — Create and fill a new temporary table named spick_expensive_bicycles with the ID, make, model and price of bicycles priced above 4000. Use SELECT INTO and display the new table
- `SELECT ... INTO TEMP` erstellt eine neue temporäre Tabelle und befüllt sie direkt mit den Fahrrädern über 4000, die Quelldaten bleiben unverändert
- Format: `SELECT spalte1, spalte2 INTO TEMP neue_tabelle FROM quelltabelle WHERE bedingung`, spätere Änderungen der Quelle werden nicht automatisch übernommen

In [57]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    SELECT bicycleid, make, model, price                                        --Gewünschte Quellspalten auswählen
    INTO TEMP spick_expensive_bicycles                                          --Neue temporäre Tabelle erstellen und mit dem Abfrageergebnis füllen
    FROM bicycles                                                               --Originaltabelle nur lesen
    WHERE price > 4000;                                                         --Nur teure Fahrräder übernehmen, spätere Quelländerungen aktualisieren die Kopie nicht
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                    --Alle Spalten der neuen Tabelle anzeigen
    FROM spick_expensive_bicycles                                               --Neu erstellte Tabelle verwenden
    ORDER BY price DESC, bicycleid;                                             --Teuerste Fahrräder zuerst, temporäre Tabelle endet mit dieser Verbindung
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,price
0,C187CGZK39,Trek,Marlin,7995.90
1,CENB0TLQKE,Bianchi,Oltre,7989.74
2,CJSLUUOM7Z,Santa Cruz,Hightower,7984.31
3,CLQCGBXCAG,Merida,Scultura,7972.88
4,CTVKH7S3PZ,Trek,Marlin,7972.58
...,...,...,...,...
1110,CSZKS5TUH1,Trek,Domane,4008.82
1111,C0TABSD96T,Bianchi,Oltre,4007.85
1112,CKT2D7A370,Specialized,Rockhopper,4005.70
1113,CFNOL6I7Z9,Scott,Addict,4004.32


### ALTER ADD COLUMN — Create a temporary copy of bicycles and add an empty color column with type VARCHAR(30). Display five rows including the new column
- `ALTER TABLE ... ADD COLUMN color VARCHAR(30)` ergänzt eine Textspalte mit maximal 30 Zeichen, bestehende Zeilen erhalten ohne DEFAULT zunächst NULL
- Format: `ALTER TABLE tabelle ADD COLUMN neue_spalte datentyp`, die bisherigen Spalten und Daten bleiben erhalten

In [58]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                          --Temporäre Übungstabelle erstellen, beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status                 --Originalspalten übernehmen
    FROM bicycles;                                                              --Originaldaten kopieren, sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                                  --Struktur der temporären Tabelle ändern
    ADD COLUMN color VARCHAR(30);                                               --Neue Textspalte ergänzen, ohne DEFAULT sind bestehende Zeilen dort NULL
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model, color                                        --Neue Spalte zusammen mit bestehenden Daten anzeigen
    FROM spick_bicycles                                                         --Temporäre Kopie verwenden
    ORDER BY bicycleid                                                          --Stabile Reihenfolge festlegen
    LIMIT 5;                                                                    --Fünf Beispielzeilen anzeigen
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,color
0,C02MOC0YLP,Cube,Reaction,None
1,C037S02BZX,Cannondale,SuperSix,None
2,C03X2W8T7W,Cannondale,Synapse,None
3,C04CCUTU6Z,Scott,Spark,None
4,C04U0GF99X,Merida,Big Nine,None


### ALTER COLUMN TYPE — Create a temporary copy of bicycles and change the stored year column to TEXT using an explicit conversion. Display five bicycle IDs, years and their resulting PostgreSQL data type
- `ALTER COLUMN year TYPE TEXT USING year::TEXT` ändert den gespeicherten Spaltentyp in der temporären Kopie, `pg_typeof(year)` zeigt den resultierenden Datentyp
- Format: `ALTER TABLE tabelle ALTER COLUMN spalte TYPE neuer_datentyp USING spalte::neuer_datentyp`, anders als CAST im SELECT wird die Tabellenstruktur geändert

In [59]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                          --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status                 --Originalspalten übernehmen
    FROM bicycles;                                                              --Originaldaten kopieren, sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                                  --Struktur der Übungskopie verändern
    ALTER COLUMN year TYPE TEXT USING year::TEXT;                               --Gespeicherten Spaltentyp ändern, USING legt die Konvertierung fest
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, year, pg_typeof(year)::TEXT AS year_type                  --Werte und tatsächlichen PostgreSQL-Datentyp kontrollieren
    FROM spick_bicycles                                                         --Temporäre Kopie verwenden
    ORDER BY bicycleid                                                          --Stabile Reihenfolge festlegen
    LIMIT 5;                                                                    --Fünf Beispielzeilen anzeigen
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,year,year_type
0,C02MOC0YLP,2020,text
1,C037S02BZX,2023,text
2,C03X2W8T7W,2022,text
3,C04CCUTU6Z,2020,text
4,C04U0GF99X,2021,text


### ALTER VARCHAR length — Create a temporary copy of bicycles with model initially stored as VARCHAR(100). Increase the maximum length to 200 characters and display five rows
- `ALTER COLUMN model TYPE VARCHAR(200)` erhöht die erlaubte Textlänge von 100 auf 200 Zeichen, vorhandene Modellnamen bleiben unverändert
- Format: `ALTER TABLE tabelle ALTER COLUMN spalte TYPE VARCHAR(neue_laenge)`, beim Verkleinern müssen vorhandene Texte in die neue Länge passen

In [60]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                          --Temporäre Übungstabelle erstellen, beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status                 --Originalspalten übernehmen
    FROM bicycles;                                                              --Originaldaten kopieren, sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                                  --Kopie zunächst auf einen begrenzten Texttyp setzen
    ALTER COLUMN model TYPE VARCHAR(100);                                       --Aus dem ursprünglichen TEXT wird Text mit maximal 100 Zeichen
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                                  --Struktur der Übungskopie erneut ändern
    ALTER COLUMN model TYPE VARCHAR(200);                                       --Erlaubte Textlänge auf 200 Zeichen erhöhen; bestehende Werte bleiben erhalten
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT bicycleid, make, model                                               --Bestehende Werte nach der Strukturänderung anzeigen
    FROM spick_bicycles                                                         --Übungskopie abfragen
    ORDER BY bicycleid                                                          --Nach ID sortieren
    LIMIT 5;                                                                    --Fünf Beispielzeilen anzeigen
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model
0,C02MOC0YLP,Cube,Reaction
1,C037S02BZX,Cannondale,SuperSix
2,C03X2W8T7W,Cannondale,Synapse
3,C04CCUTU6Z,Scott,Spark
4,C04U0GF99X,Merida,Big Nine


### ALTER DROP COLUMN — Create a temporary copy of bicycles, add a color column and then remove that column. Display five rows of the remaining table
- `ALTER TABLE ... DROP COLUMN color` entfernt die Farbspalte inkl. ihrer gespeicherten Werte, alle übrigen Spalten bleiben erhalten
- Format: `ALTER TABLE tabelle DROP COLUMN spalte`, entfernt eine ganze Spalte und keine einzelnen Zeilen

In [61]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                          --Temporäre Übungstabelle erstellen, beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status                 --Originalspalten übernehmen
    FROM bicycles;                                                              --Originaldaten kopieren, sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                                  --Übungskopie verändern
    ADD COLUMN color VARCHAR(30);                                               --Spalte für die Löschübung zuerst erstellen
    """))

    connection.execute(text("""
    ALTER TABLE spick_bicycles                                                  --Tabelle bleibt bestehen
    DROP COLUMN color;                                                          --Nur die Spalte color und ihre gespeicherten Werte entfernen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT *                                                                    --Verbleibende Spalten anzeigen; color fehlt
    FROM spick_bicycles                                                         --Übungskopie abfragen
    ORDER BY bicycleid                                                          --Nach ID sortieren
    LIMIT 5;                                                                    --Fünf Beispielzeilen anzeigen
    """, con=connection)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,year,framenr,price,status
0,C02MOC0YLP,Cube,Reaction,2020,XMD5040,4823.86,Sold
1,C037S02BZX,Cannondale,SuperSix,2023,JRN 233,2424.03,Sold
2,C03X2W8T7W,Cannondale,Synapse,2022,950 OPV,3114.65,In Maintenance
3,C04CCUTU6Z,Scott,Spark,2020,72Y TB8,6276.01,Sold
4,C04U0GF99X,Merida,Big Nine,2021,0019,6150.43,Sold


### DROP TABLE — Create a temporary copy of bicycles, drop the copy and verify that the temporary table no longer exists
- `DROP TABLE` entfernt die temporäre Kopie vollständig inkl. Daten und Tabellenstruktur, die Originaltabelle bleibt erhalten
- Format: `DROP TABLE tabelle`, anders als bei DELETE bleibt keine leere Tabelle bestehen

In [62]:
engine = create_engine(db_connection_url)

with engine.begin() as connection:
    connection.execute(text("""
    CREATE TEMP TABLE spick_bicycles ON COMMIT DROP AS                          --Temporäre Übungstabelle erstellen; beim Transaktionsende automatisch löschen
    SELECT bicycleid, make, model, year, framenr, price, status                 --Originalspalten übernehmen
    FROM bicycles;                                                              --Originaldaten kopieren, sämtliche folgenden Änderungen betreffen nur die Kopie
    """))

    connection.execute(text("""
    DROP TABLE spick_bicycles;                                                  --Übungstabelle vollständig inklusive Struktur und Daten löschen
    """))

    df_spickap01 = pd.read_sql_query("""
    SELECT to_regclass('pg_temp.spick_bicycles') IS NULL AS table_removed;      --TRUE bestätigt das Fehlen der temporären Tabelle, bicycles bleibt unverändert
    """, con=connection)

engine.dispose()
display(df_spickap01)

,table_removed
0,True
